# Differential Geometries and Measurable Modes

Notebook 23 showed that differential readout removes some disturbances and preserves others.

Notebook 29 asks:

**How does sensor arrangement determine which disturbances are removable?**

A measurement geometry acts like a spatial filter. Different sensor layouts reject different spatial modes and preserve different signals.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIGURES = ROOT / "figures"
DATA = ROOT / "data"

FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

rng = np.random.default_rng(42)


## Simulated disturbance field

We begin with a one-dimensional disturbance field.

It contains long-, medium-, and short-wavelength components.

Long-wavelength structure is more common across distant sensors. Short-wavelength structure changes more rapidly with position.


In [ ]:
x = np.linspace(0, 100, 1000)

field = (
    1.0 * np.sin(2 * np.pi * x / 120)
    + 0.5 * np.sin(2 * np.pi * x / 40)
    + 0.2 * np.sin(2 * np.pi * x / 10)
)

sensor_positions = {
    "A": 10,
    "B": 40,
    "C": 70,
    "D": 100
}


def field_at(position, wavelength=None):
    if wavelength is None:
        return np.interp(position, x, field)
    return np.sin(2 * np.pi * position / wavelength)


In [ ]:
plt.figure(figsize=(9, 4))

plt.plot(x, field, label="disturbance field")

for name, pos in sensor_positions.items():
    plt.axvline(pos, linestyle="--", alpha=0.5)
    plt.text(pos, field_at(pos) + 0.15, name, ha="center")

plt.xlabel("position")
plt.ylabel("field amplitude")

plt.title("Spatial disturbance field sampled by sensors")

plt.tight_layout()

plt.savefig(
    FIGURES / "29_spatial_field.png",
    dpi=200
)

plt.show()


## Geometry responses

We compare four measurement geometries.

- Single sensor: \(A\)
- Pair: \(A - B\)
- Triple baseline: \(A - 2B + C\)
- Square-like four-point mode: \(A - B - C + D\)

These are not the only possible geometries. They are simple examples showing how different layouts filter spatial structure.


In [ ]:
A = sensor_positions["A"]
B = sensor_positions["B"]
C = sensor_positions["C"]
D = sensor_positions["D"]

responses = {
    "single": field_at(A),
    "pair": field_at(A) - field_at(B),
    "triple": field_at(A) - 2 * field_at(B) + field_at(C),
    "four-point": field_at(A) - field_at(B) - field_at(C) + field_at(D)
}

response_df = pd.DataFrame({
    "geometry": list(responses.keys()),
    "response": list(responses.values()),
    "absolute_response": [abs(v) for v in responses.values()]
})

response_df


In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    response_df["geometry"],
    response_df["absolute_response"]
)

plt.ylabel("Absolute response")
plt.title("Sensor geometry determines response to a field")

plt.tight_layout()

plt.savefig(
    FIGURES / "29_geometry_response.png",
    dpi=200
)

plt.show()


## Wavelength dependence

Now we sweep the wavelength of a pure sinusoidal disturbance.

For each wavelength, we evaluate the response of each geometry.

This shows which spatial modes are rejected and which survive measurement.


In [ ]:
wavelengths = np.linspace(5, 200, 160)

rows = []

for wavelength in wavelengths:

    values = {
        "A": field_at(A, wavelength),
        "B": field_at(B, wavelength),
        "C": field_at(C, wavelength),
        "D": field_at(D, wavelength)
    }

    geometry_values = {
        "single": values["A"],
        "pair": values["A"] - values["B"],
        "triple": values["A"] - 2 * values["B"] + values["C"],
        "four-point": values["A"] - values["B"] - values["C"] + values["D"]
    }

    for geometry, response in geometry_values.items():
        rows.append({
            "wavelength": wavelength,
            "geometry": geometry,
            "response": response,
            "absolute_response": abs(response)
        })

sweep = pd.DataFrame(rows)


In [ ]:
pair = sweep[sweep["geometry"] == "pair"]

plt.figure(figsize=(8, 4))

plt.plot(
    pair["wavelength"],
    pair["absolute_response"]
)

plt.xlabel("Disturbance wavelength")
plt.ylabel("Absolute pair response")

plt.title("Pair geometry rejects very long-wavelength structure")

plt.tight_layout()

plt.savefig(
    FIGURES / "29_pair_wavelength_response.png",
    dpi=200
)

plt.show()


In [ ]:
plt.figure(figsize=(9, 5))

for geometry, group in sweep.groupby("geometry"):
    plt.plot(
        group["wavelength"],
        group["absolute_response"],
        label=geometry
    )

plt.xlabel("Disturbance wavelength")
plt.ylabel("Absolute response")

plt.title("Different geometries measure different spatial modes")

plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "29_geometry_comparison.png",
    dpi=200
)

plt.show()


## Recoverability summary

The geometry determines which modes are visible.

Long-wavelength common structure tends to cancel in differential geometries. Faster spatial variation survives more strongly.


In [ ]:
summary = pd.DataFrame({
    "geometry": [
        "single",
        "pair",
        "triple",
        "four-point"
    ],
    "measurement": [
        "field value",
        "first difference",
        "second difference",
        "four-point contrast"
    ],
    "best_for": [
        "local signal",
        "differential signal",
        "curvature-like structure",
        "higher-order spatial contrast"
    ],
    "rejects": [
        "nothing",
        "common offset",
        "common offset and linear trend",
        "selected common and gradient-like modes"
    ]
})

summary.to_csv(
    DATA / "29_geometry_summary.csv",
    index=False
)

summary


# Lesson

Differential sensing is not only about noise cancellation.

Sensor geometry determines which spatial modes are visible.

Different arrangements reject different kinds of common structure.

Geometry creates context.

Context determines which signals survive measurement.

Notebook 37 will connect this toy geometry directly to an atom-interferometer-style differential readout.
